# Path 5: Conflict responses (409)

Checks every 409 the API returns:

- Duplicate customer email and duplicate product SKU (`unique_violation`) on create and update.
- An order or line item that points at a missing row (`foreign_key_violation`).
- Deleting a product that an order-item still references. That guard answers before the database, so the body is the product message rather than `foreign_key_violation`.

Database conflicts respond with `{ "error": "Conflict", "reason": "..." }` and do not include the conflicting email or SKU.

Run top-to-bottom (e.g. `jupyter nbconvert --to notebook --execute 05_conflict_responses.ipynb`). The last cell deletes the rows this notebook created.

In [ ]:
import os

import requests
from faker import Faker

BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:3002/api")
fake = Faker()
NONEXISTENT_ID = 999_999_999


def assert_db_conflict(resp, reason, label):
    assert resp.status_code == 409, (
        f"{label}: expected 409, got {resp.status_code} {resp.text}"
    )
    body = resp.json()
    assert body == {"error": "Conflict", "reason": reason}, f"{label}: {body}"
    print(f"Confirmed 409 {reason}: {label}")


print(f"BASE_URL={BASE_URL}")

## Step 1 - Create two customers and two products

In [ ]:
def create_customer():
    resp = requests.post(
        f"{BASE_URL}/customers",
        json={"email": fake.unique.email(), "password": fake.password(length=14)},
    )
    assert resp.status_code == 201, resp.text
    return resp.json()


def create_product():
    resp = requests.post(
        f"{BASE_URL}/products",
        json={
            "sku": f"SKU-{fake.unique.bothify(text='???-####').upper()}",
            "name": fake.unique.catch_phrase(),
            "unit_price_cents": fake.random_int(min=99, max=14999),
        },
    )
    assert resp.status_code == 201, resp.text
    return resp.json()


customer_a = create_customer()
customer_b = create_customer()
product_a = create_product()
product_b = create_product()

print(
    f"customers={customer_a['id']},{customer_b['id']} "
    f"products={product_a['id']},{product_b['id']}"
)

## Step 2 - Duplicate customer email returns 409 unique_violation

In [ ]:
resp = requests.post(
    f"{BASE_URL}/customers",
    json={"email": customer_a["email"], "password": fake.password(length=14)},
)
assert_db_conflict(resp, "unique_violation", "POST /customers with an existing email")

resp = requests.put(
    f"{BASE_URL}/customers/{customer_b['id']}",
    json={"email": customer_a["email"], "password": fake.password(length=14)},
)
assert_db_conflict(resp, "unique_violation", "PUT /customers/:id onto an existing email")

resp = requests.patch(
    f"{BASE_URL}/customers/{customer_b['id']}",
    json={"email": customer_a["email"]},
)
assert_db_conflict(resp, "unique_violation", "PATCH /customers/:id onto an existing email")

## Step 3 - Duplicate product SKU returns 409 unique_violation

In [ ]:
resp = requests.post(
    f"{BASE_URL}/products",
    json={
        "sku": product_a["sku"],
        "name": fake.unique.catch_phrase(),
        "unit_price_cents": 100,
    },
)
assert_db_conflict(resp, "unique_violation", "POST /products with an existing sku")

resp = requests.put(
    f"{BASE_URL}/products/{product_b['id']}",
    json={
        "sku": product_a["sku"],
        "name": product_b["name"],
        "unit_price_cents": product_b["unit_price_cents"],
    },
)
assert_db_conflict(resp, "unique_violation", "PUT /products/:id onto an existing sku")

resp = requests.patch(
    f"{BASE_URL}/products/{product_b['id']}",
    json={"sku": product_a["sku"]},
)
assert_db_conflict(resp, "unique_violation", "PATCH /products/:id onto an existing sku")

## Step 4 - Missing foreign keys return 409 foreign_key_violation

In [ ]:
resp = requests.post(
    f"{BASE_URL}/orders",
    json={"customer_id": NONEXISTENT_ID},
)
assert_db_conflict(resp, "foreign_key_violation", "POST /orders for a missing customer")

resp = requests.post(f"{BASE_URL}/orders", json={"customer_id": customer_a["id"]})
assert resp.status_code == 201, resp.text
order = resp.json()

resp = requests.put(
    f"{BASE_URL}/orders/{order['id']}",
    json={"customer_id": NONEXISTENT_ID},
)
assert_db_conflict(resp, "foreign_key_violation", "PUT /orders/:id onto a missing customer")

resp = requests.post(
    f"{BASE_URL}/order-items",
    json={
        "order_id": NONEXISTENT_ID,
        "product_id": product_a["id"],
        "quantity": 1,
        "unit_price_cents": product_a["unit_price_cents"],
    },
)
assert_db_conflict(resp, "foreign_key_violation", "POST /order-items for a missing order")

resp = requests.post(
    f"{BASE_URL}/order-items",
    json={
        "order_id": order["id"],
        "product_id": NONEXISTENT_ID,
        "quantity": 1,
        "unit_price_cents": product_a["unit_price_cents"],
    },
)
assert_db_conflict(resp, "foreign_key_violation", "POST /order-items for a missing product")

## Step 5 - DELETE /products/:id while an order-item references it returns 409

In [ ]:
resp = requests.post(
    f"{BASE_URL}/order-items",
    json={
        "order_id": order["id"],
        "product_id": product_a["id"],
        "quantity": 1,
        "unit_price_cents": product_a["unit_price_cents"],
    },
)
assert resp.status_code == 201, resp.text
item = resp.json()

resp = requests.delete(f"{BASE_URL}/products/{product_a['id']}")
assert resp.status_code == 409, f"expected 409, got {resp.status_code} {resp.text}"
assert resp.json() == {"error": "Product is referenced by existing order items"}, resp.json()
print("Confirmed 409: product cannot be deleted while an order-item references it")

## Step 6 - Remove the rows this notebook created

In [ ]:
resp = requests.delete(f"{BASE_URL}/order-items/{item['id']}")
assert resp.status_code == 200, resp.text

for product in (product_a, product_b):
    resp = requests.delete(f"{BASE_URL}/products/{product['id']}")
    assert resp.status_code == 200, resp.text

resp = requests.delete(f"{BASE_URL}/orders/{order['id']}")
assert resp.status_code == 200, resp.text

for customer in (customer_a, customer_b):
    resp = requests.delete(f"{BASE_URL}/customers/{customer['id']}")
    assert resp.status_code == 200, resp.text

print("Removed the customers, products, order, and order-item created above")